# Steering Qwen3-8B with an emotion vector (smoke test)

Adds the **desperate** vector at block 23 while Qwen3-8B writes, at strengths 0, 0.25, 0.5, 1.0 and 1.5, with the same prompt and the same seed. Strength is a fraction of the residual stream's average size at block 23 (the paper used 0.5).

**Run on:** an NVIDIA GPU (tested on a Brev A4000, 16 GB). Run it from inside `notebooks/`.

**Needs:**
- `emotion_vectors.npz` and `manifest.json` from extraction run `ab1333f49024705c` (Drive link in `emotion_vector_pipeline/README.md`), extracted into `emotion_vector_pipeline/outputs/emotion_extraction/ab1333f49024705c/`
- the unzipped story dataset (see the main README, "Data")

In [1]:
import json, sys
from pathlib import Path
import torch

#let the notebook import our code from src/
sys.path.insert(0, str(Path.cwd().parent))
from src.steering import LAYER, MODEL_REVISION, load_model_8b, load_vector, residual_norms, steer, generate

RUN_DIR = Path("../emotion_vector_pipeline/outputs/emotion_extraction/ab1333f49024705c")
STORIES = Path("../emotion_vector_pipeline/data/emotion_stories_qwen_full1120.jsonl")
OUT = Path("../results/steering")
OUT.mkdir(parents=True, exist_ok=True)
print(torch.cuda.get_device_name(0))

NVIDIA RTX A4000


## 1. Load the model and the vector
Same revision and 4-bit setup as the extraction notebook, so the vector matches the model.

In [2]:
#the vectors must come from the same model revision we load
manifest = json.loads((RUN_DIR / "manifest.json").read_text())
assert manifest["config"]["model_revision"] == MODEL_REVISION

tok, model = load_model_8b()
print(len(model.model.layers), model.config.hidden_size)  #expect 36 4096
print(round(model.get_memory_footprint() / 2**30, 2), "GiB on the GPU")

v = load_vector(RUN_DIR / "emotion_vectors.npz", "desperate")
print(v.shape, round(v.norm().item(), 4))  #expect torch.Size([4096]) 1.0

`torch_dtype` is deprecated! Use `dtype` instead!


Loading checkpoint shards:   0%|          | 0/5 [00:00<?, ?it/s]

36 4096
5.55 GiB on the GPU
torch.Size([4096]) 1.0


## 2. How big is the residual stream at block 23?
Strength 0.5 means "half the usual size of the residual stream", so we measure the usual size on 28 validation stories (2 per emotion).

In [3]:
rows = [json.loads(line) for line in STORIES.open()]
texts = []
for e in sorted({r["emotion"] for r in rows}):
    texts += [r["text"] for r in rows if r["emotion"] == e and r["split"] == "validation"][:2]

first, rest = residual_norms(model, tok, texts)
NORM = rest.mean().item()
print(len(texts), "stories,", len(rest), "tokens")
print("first token, mean norm:", round(first.mean().item(), 1))
print("other tokens, mean norm:", round(NORM, 1))

28 stories, 3580 tokens
first token, mean norm: 11737.6
other tokens, mean norm: 198.0


## 3. Generate at each strength
Same prompt, same seed. Only the strength changes.

In [4]:
PROMPT = "Write a short paragraph about your morning."
STRENGTHS = [0, 0.25, 0.5, 1.0, 1.5]
SEED = 0

baseline = generate(model, tok, PROMPT, seed=SEED)  #no hook at all
outputs = {}
for c in STRENGTHS:
    with steer(model, v, c, NORM):
        outputs[c] = generate(model, tok, PROMPT, seed=SEED)
    print(f"--- strength {c} ---\n{outputs[c]}\n")

--- strength 0 ---
My morning starts with the soft glow of dawn filtering through the curtains. I brew a cup of coffee, savoring the quiet moment as I listen to the gentle sounds of the city waking up. I take a few deep breaths, stretch, and let the calm of the early hours settle into my mind. It's a peaceful ritual that sets a positive tone for the day ahead.



--- strength 0.25 ---
Every morning, I wake up at 7 a.m. and immediately check my phone to see what the weather is like. If it's raining, I take a coffee break before heading to work. If it's sunny, I try to get outside for a quick run. I usually have a cup of coffee, brush my teeth, and then get dressed. My day starts with a lot of stress



--- strength 0.5 ---
I woke up at 7 a.m. and tried to stay asleep for another hour. I got up, made coffee, and tried to think of something else. I took a shower, got dressed, and tried to remember what day it was. I got out of the house and tried to get to work. I got to work and tried to remember why I was here. I got to work



--- strength 1.0 ---
I tried to get up at 6:00. I tried to get up at 6:00. I tried to get up at 6:00. I tried to get up at 6:00. I tried to get up at 6:00. I tried to get up at 6:00. I tried to get up at 6



--- strength 1.5 ---
I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting. I'm getting.



## 4. Checks

In [5]:
#check 1: strength 0 adds a zero vector, so it must match the run with no hook
assert outputs[0] == baseline, "strength 0 changed the output"
print("check 1 passed: strength 0 == no hook")

#check 2: the hook adds exactly strength x NORM to every token at block 23
captured = {}
def grab(module, args, output):
    captured["h"] = (output[0] if isinstance(output, tuple) else output).float()

ids = tok(PROMPT, return_tensors="pt").input_ids.to(model.device)
with torch.inference_mode():
    handle = model.model.layers[LAYER].register_forward_hook(grab)
    model(ids)
    handle.remove()
    base_h = captured["h"]
    with steer(model, v, 0.5, NORM):
        #registered after the steering hook, so it sees the steered output
        handle = model.model.layers[LAYER].register_forward_hook(grab)
        model(ids)
        handle.remove()

added = (captured["h"] - base_h)[0].norm(dim=-1)
print("added norm per token:", round(added.min().item(), 2), "to", round(added.max().item(), 2),
      "| expected:", round(0.5 * NORM, 2))
assert torch.allclose(added, torch.full_like(added, 0.5 * NORM), rtol=0.05)
print("check 2 passed")

check 1 passed: strength 0 == no hook
added norm per token: 99.01 to 99.15 | expected: 99.02
check 2 passed


## 5. Save

In [6]:
record = dict(prompt=PROMPT, seed=SEED, emotion="desperate", layer=LAYER, residual_norm=NORM,
              model_revision=MODEL_REVISION, baseline=baseline,
              outputs={str(c): text for c, text in outputs.items()})
(OUT / "smoke_desperate.json").write_text(json.dumps(record, indent=2))
print("saved", OUT / "smoke_desperate.json")

saved ../results/steering/smoke_desperate.json
